In [ ]:
import fitz
from pathlib import Path

pdfs = sorted(Path(r"C:\dev\paper-rag\data\pdfs").glob("*.pdf"))
print(len(pdfs), "PDFs")

for p in pdfs:
    doc = fitz.open(p)
    chars = sum(len(page.get_text()) for page in doc)
    print(f"{p.name:145} pages={len(doc):3}  chars={chars:7}  title={doc.metadata.get('title')!r}")

In [ ]:
doc = fitz.open(pdfs[0])
print(doc[0].get_text()[:1500])

In [ ]:
from pathlib import Path
from paper_rag.ingest import parse_pdf, chunk_document

docs = [parse_pdf(p) for p in sorted (Path(r"C:\dev\paper-rag\data\pdfs").glob("*.pdf"))]
for d in docs:
    print(f"{d.title[:60]:60} pages={d.page_count:3} chunks={len(chunk_document(d)):4}")

In [ ]:
d = next(x for x in docs if x.file_name.startswith("CIC_2024"))
for c in chunk_document(d)[:3]:
    print(c.chunk_id, "pages", c.page_start, "-", c.page_end, "\n", c.text, "\n---")

In [ ]:
import random

for d in docs:
    cs = chunk_document(d)
    print(f"{d.title[:50]:50} last chunk on page {cs[-1].page_end:2} of {d.page_count}")

In [ ]:
short = [c for d in docs for c in chunk_document(d) if len(c.text) < 120]
print(len(short), "short chunks")
for c in random.sample(short, min(8, len(short))):
    print(repr(c.text))

In [ ]:
d = next(x for x in docs if x.title.startswith("Wavelet"))
for c in random.sample(chunk_document(d), 3):
    print("page", c.page_start, "\n", c.text, "\n---")

In [ ]:
import re
from collections import Counter
for d in docs:
    first = d.pages[0].text if d.pages else ""
    years = Counter(re.findall(r"\b(19[89]\d|20[0-3]\d)\b", first))
    print(f"{d.file_name[:55]:55} {years.most_common(3)}")